# Launch & finish DPS runs of `s1-seasonal-rtc` (Sentinel-1 seasonal composites)

Submits one DPS job per **boreal tile × year** to build the 12-band seasonal Sentinel-1 composite (frozen / summer / shoulder
VV and VH means + valid-pass counts, OPERA RTC-S1, Refined Lee), finds which tiles are still **missing** or **failed**, and
resubmits only those. After jobs finish, it builds the tile index (tindex) that defines what "complete" means and that downstream
runs (e.g. `run_boreal_biomass_map` with `'sar'` inputs) read.

Same layout as `icesat2_boreal/notebooks/3.Gridded_product_development/DPS_run_boreal_v2.ipynb`.

**Run top to bottom. The only cell you edit is `1. Run configuration`.** Nothing is submitted unless `DRY_RUN = False`.

| Step | What it does | Touches DPS? |
|---|---|---|
| 2. Status | Reads each year's tindex: tiles **have** vs **missing** | no |
| 3. Fails | Latest job per tile from saved submission logs → which tiles **failed** (and are still not in the tindex) | reads job status |
| 4. Coverage | Optional: OPERA RTC-S1 granule count per tile and year (CMR); tiles with none are not runnable. Cached. | no (CMR only) |
| 5. Plan | Per year: tiles to submit (`DO_MISSING` / `DO_FAILS`), minus tiles that cannot run. Saved as CSV. | no |
| 6. Submit | Submits the plan (only if `DRY_RUN = False`) | **yes** |
| 7. Assess | Status counts for the submission just made | reads job status |
| 8. Build tindex | After jobs finish: backs up, rebuilds from **all** DPS outputs for the identifier, verifies the count did not shrink | no |

**Typical cycle**: `DO_MISSING=True` → plan → submit → wait → 7 → 8 → back to 2. Then `DO_FAILS=True` for stragglers.

**`DO_MISSING` vs `DO_FAILS`**: as in the boreal runner. MISSING = in `TILE_SET` but not in the tindex (only as current as the last
step 8). FAILS = the tile's most recent job for this identifier failed (or never got a job id) and the tile is not in the tindex.
Neither: every tile in `TILE_SET` (needs `CONFIRM_FULL_RUN = True`).

**Tile grid.** Each job gets its tile's bounds in the boreal tile CRS, which is identical to `custom_albers.wkt` (checked in
section 1b), with `crs = bbox_crs = @custom_albers.wkt`, so outputs line up with the HLS/topo stacks. The compositor needs the bbox
to span whole pixels: bounds are rounded to 1 cm at the upper-left corner and the width/height snapped to whole 30 m pixels.

**Platform:** MAAP ADE, kernel with `maap-py` (the registration/submission interface of SDK v4.2.0 used by this algorithm).
Notebook and repo in `/projects` (home), outputs in the bucket.

In [ ]:
import os, sys, glob, json, shutil, datetime, requests
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
import pandas as pd
import geopandas as gpd
import pyproj

from maap.maap import MAAP
maap = MAAP(maap_host='api.maap-project.org')

S1_REPO_PATH = '/projects/code/s1-composite'          # this repo (custom_albers.wkt, algorithm_config.yml)

## 1. Run configuration — the only cell to edit
Fill-in runs **must** use the same `VERSION`, `SUB_IDENTIFIER` and `ALGO_PARAMS` as the original run, so new tiles match the rest.

In [ ]:
# --- What product -----------------------------------------------------------
ALGO_ID      = 's1-seasonal-rtc'            # algorithm_name in algorithm_config.yml
VERSION      = 'main'                       # algorithm_version (branch) registered on MAAP
USER         = 'montesano'                  # recorded in the submission logs; DPS outputs go to this user's bucket
WORKER_TYPE  = 'maap-dps-worker-16gb'       # queue in algorithm_config.yml
SUB_IDENTIFIER = 'opera_rtc_refinedlee_30m' # part of the DPS identifier: S1_<year>/<SUB_IDENTIFIER>
YEAR_LIST    = ['2020']

# --- Which tiles -------------------------------------------------------------
DO_MISSING = True        # tiles in TILE_SET not yet in the tindex
DO_FAILS   = False       # tiles whose latest job failed and are still not in the tindex
CONFIRM_FULL_RUN = False # must be True to submit every tile in TILE_SET (both flags False)
TILE_SET = None          # None = all boreal tiles; or a list of tile_num, e.g. [3364]
REFRESH_FAILS = True     # re-query job status (slow for many jobs); False = reuse the cached fails JSON
CHECK_OPERA_COVERAGE = True   # mark tiles with no OPERA RTC-S1 granules that year as not runnable (CMR, cached)

# --- Safety -----------------------------------------------------------------
DRY_RUN = True           # True: build and save the plan only. False: submit in step 6.
MAX_JOBS = 6000          # refuse to submit more than this in one go

# --- Algorithm parameters (positional inputs 4-10 of run.sh; bbox, year, tile_identifier are set per tile) ---
ALGO_PARAMS = {
    'crs': '@custom_albers.wkt',            # output CRS = boreal tile CRS
    'bbox_crs': '@custom_albers.wkt',       # bbox is given in tile CRS metres -> exact tile grid
    'spacing': '30',
    'resampling': 'nearest',
    'speckle_filter': 'true',
    'season_windows': '{}',                 # standard frozen / summer / shoulder windows
    'write_stac': 'false',
}

# --- Paths --------------------------------------------------------------------
BOREAL_TILES_FN = '/projects/shared-buckets/montesano/databank/boreal_tiles_v004_model_ready.gpkg'
DPS_SUBMISSION_RESULTS_DIR = '/projects/my-public-bucket/dps_submission_results'
TINDEX_ROOT     = '/projects/my-public-bucket/DPS_tile_lists/S1_COMPOSITE'
DPS_OUTPUT_ROOT = f'/projects/my-private-bucket/dps_output/{ALGO_ID}/{VERSION}'
PLAN_DIR        = f'{DPS_SUBMISSION_RESULTS_DIR}/plans'
for d in (DPS_SUBMISSION_RESULTS_DIR, PLAN_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Derived names (do not edit)
TINDEX_TYPE = 'S1'
IDENTIFIERS = {y: f'S1_{y}/{SUB_IDENTIFIER}' for y in YEAR_LIST}
MODE = '+'.join([m for m, on in (('MISSING', DO_MISSING), ('FAILS', DO_FAILS)) if on]) or 'ALL'
SPACING = float(ALGO_PARAMS['spacing'])
print(f'{ALGO_ID} {VERSION} | years {YEAR_LIST[0]}–{YEAR_LIST[-1]} | mode {MODE} | DRY_RUN={DRY_RUN}')
for y, i in IDENTIFIERS.items():
    print('  ', i)

### 1b. Tile grid check: boreal tile CRS must equal `custom_albers.wkt`

In [ ]:
boreal_tiles = gpd.read_file(BOREAL_TILES_FN)
boreal_tiles['tile_num'] = boreal_tiles.tile_num.astype(int)
albers = pyproj.CRS(open(f'{S1_REPO_PATH}/custom_albers.wkt').read())
assert boreal_tiles.crs.equals(albers, ignore_axis_order=True), 'tile CRS differs from custom_albers.wkt: pass bbox_crs/crs explicitly'
print('tile CRS == custom_albers.wkt:', True, '|', len(boreal_tiles), 'tiles')

### Helper functions (no edits needed)
Per-tile parameters: the tile's bounds as the bbox (snapped to whole pixels), the year, and the tile number as `tile_identifier`
(output file `SAR_S1_<year>_<tile>.tif`).

In [ ]:
TILE_GEOM = boreal_tiles.set_index('tile_num').geometry
TILE_LL = boreal_tiles.to_crs(4326).set_index('tile_num').geometry

def _fmt(v):
    return f'{v:.2f}'.rstrip('0').rstrip('.')

def tile_bbox(tile_num, spacing=SPACING):
    """Tile bounds in tile CRS, rounded to 1 cm at the upper-left corner, width/height snapped to whole pixels."""
    xmin, ymin, xmax, ymax = TILE_GEOM.loc[tile_num].bounds
    xmin, ymax = round(xmin, 2), round(ymax, 2)
    nx, ny = round((xmax - xmin) / spacing), round((ymax - ymin) / spacing)
    if nx < 1 or ny < 1:
        raise ValueError(f'tile narrower than one pixel ({nx} x {ny})')
    return (xmin, ymax - ny * spacing, xmin + nx * spacing, ymax), (nx, ny)

def crosses_dateline(tile_num):
    w, s, e, n = TILE_LL.loc[tile_num].bounds
    return (e - w) > 180

def get_in_param_dict(tile_num, year):
    """Inputs for submitJob: bbox (one string), year, tile_identifier + ALGO_PARAMS."""
    if crosses_dateline(tile_num):
        raise ValueError('crosses the antimeridian in EPSG:4326 (compositor: split the tile)')
    (xmin, ymin, xmax, ymax), _ = tile_bbox(tile_num)
    return {'bbox': ','.join(_fmt(v) for v in (xmin, ymin, xmax, ymax)), 'year': str(year),
            'tile_identifier': str(tile_num), **ALGO_PARAMS}

In [ ]:
def tindex_paths(identifier):
    d = f'{TINDEX_ROOT}/{VERSION}/{identifier}'
    return {'dir': d, 'csv': f'{d}/{TINDEX_TYPE}_tindex_master.csv', 'gpkg': f'{d}/{TINDEX_TYPE}_tindex_master.gpkg'}

def read_tindex_tiles(identifier):
    """Tiles present in the tindex for this identifier (empty set if no tindex yet)."""
    tp = tindex_paths(identifier)
    for fn in (tp['csv'], tp['gpkg']):
        if os.path.exists(fn):
            df = pd.read_csv(fn) if fn.endswith('.csv') else gpd.read_file(fn)
            return set(df.tile_num.astype(int))
    return set()

def submission_logs(identifier):
    """All saved submission CSVs for this identifier, oldest first (by the timestamp in the file name)."""
    s = identifier.replace('/', '_')
    fns = glob.glob(f'{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_{s}_submission_results_*.csv')
    return sorted(fns, key=lambda f: int(os.path.splitext(f)[0].split('_')[-1]))

def latest_jobs(identifier):
    """One row per tile: the most recent submission for this identifier."""
    parts = []
    for order, fn in enumerate(submission_logs(identifier)):
        d = pd.read_csv(fn)
        d['log_file'], d['log_order'] = os.path.basename(fn), order
        parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['tile_num', 'job_id', 'submit_status', 'log_file'])
    d = pd.concat(parts, ignore_index=True)
    d['tile_num'] = d.tile_num.astype(int)
    return d.sort_values(['log_order', 'dps_num']).drop_duplicates('tile_num', keep='last').reset_index(drop=True)

def _job_status(job_id):
    try:
        return maap.getJobStatus(job_id)
    except Exception as e:
        return f'Unknown ({e})'

def job_status_table(jobs, workers=30):
    """Adds a 'status' column via MAAP (threaded). Tiles that never got a job id are marked 'SubmitFailed'."""
    ok = jobs.job_id.notna() & (jobs.job_id.astype(str) != '')
    out = jobs.copy()
    out['status'] = 'SubmitFailed'
    if ok.any():
        ids = jobs.loc[ok, 'job_id'].astype(str).tolist()
        with ThreadPoolExecutor(workers) as ex:
            out.loc[ok, 'status'] = list(ex.map(_job_status, ids))
    return out

In [ ]:
def opera_hits(tile_num, year):
    """Number of OPERA_L2_RTC-S1_V1 granules over the tile in the year (CMR hit count)."""
    w, s, e, n = TILE_LL.loc[tile_num].bounds
    r = requests.get('https://cmr.earthdata.nasa.gov/search/granules.json',
                     params={'short_name': 'OPERA_L2_RTC-S1_V1', 'bounding_box': f'{w},{s},{e},{n}',
                             'temporal': f'{year}-01-01T00:00:00Z,{year}-12-31T23:59:59Z', 'page_size': 0}, timeout=60)
    r.raise_for_status()
    return int(r.headers.get('CMR-Hits', 0))

def opera_coverage(year, tiles, workers=16):
    """{tile_num: hits} for the year, cached in PLAN_DIR (only tiles not yet cached are queried)."""
    fn = f'{PLAN_DIR}/opera_rtc_s1_hits_{year}.csv'
    have = pd.read_csv(fn).set_index('tile_num').hits.to_dict() if os.path.exists(fn) else {}
    todo = [t for t in tiles if t not in have and not crosses_dateline(t)]
    if todo:
        with ThreadPoolExecutor(workers) as ex:
            have.update(dict(zip(todo, ex.map(lambda t: opera_hits(t, year), todo))))
        pd.DataFrame({'tile_num': list(have), 'hits': list(have.values())}).to_csv(fn, index=False)
    return have

def build_plan(year, identifier, tile_set, have, failed, coverage=None):
    """Tiles to submit for one identifier, with the reason each tile is (or is not) included."""
    missing = sorted(set(tile_set) - have)
    want = set()
    if DO_MISSING:
        want |= set(missing)
    if DO_FAILS:
        want |= set(failed)
    if not (DO_MISSING or DO_FAILS):
        want = set(tile_set)
    rows = []
    for t in sorted(want):
        try:
            get_in_param_dict(t, year)
            runnable, why = True, ''
            if coverage is not None and coverage.get(t, 1) == 0:
                runnable, why = False, f'no OPERA RTC-S1 granules in {year}'
        except (KeyError, ValueError) as e:
            runnable, why = False, str(e)
        rows.append({'identifier': identifier, 'year': year, 'tile_num': t,
                     'is_missing': t in missing, 'is_failed': t in failed, 'in_tindex': t in have,
                     'runnable': runnable, 'reason_not_runnable': why})
    return pd.DataFrame(rows, columns=['identifier', 'year', 'tile_num', 'is_missing', 'is_failed', 'in_tindex', 'runnable', 'reason_not_runnable'])

def submit_identifier(identifier, year, tiles):
    rows = []
    for i, t in enumerate(tiles, 1):
        try:
            r = maap.submitJob(identifier=identifier, algo_id=ALGO_ID, version=VERSION, queue=WORKER_TYPE,
                               **get_in_param_dict(t, year))
            job_id, status = r.id, r.status
        except Exception as e:
            job_id, status = None, f'error: {e}'
        rows.append({'dps_num': i, 'tile_num': t, 'submit_time': datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S'),
                     'algo_id': ALGO_ID, 'user': USER, 'worker_type': WORKER_TYPE, 'job_id': job_id,
                     'submit_status': status, 'IDENTIFIER': identifier})
        if i in (1, len(tiles)) or i % 100 == 0:
            print(f'  {i}/{len(tiles)} tile {t}: {status}')
    df = pd.DataFrame(rows)
    now = pd.Timestamp.now().strftime('%Y%m%d%H%M')
    fn = f"{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_{identifier.replace('/', '_')}_submission_results_{len(df)}_{now}.csv"
    df.to_csv(fn, index=False)
    print(f'  saved {fn}')
    return df

In [ ]:
def find_outputs(identifier, year):
    """All SAR_S1_<year>_<tile>.tif under this identifier's DPS output folders, newest file per tile."""
    root = f'{DPS_OUTPUT_ROOT}/{identifier}'
    fns = glob.glob(f'{root}/**/SAR_S1_{year}_*.tif', recursive=True)
    if not fns:
        return pd.DataFrame(columns=['tile_num', 'local_path', 's3_path', 'mtime'])
    df = pd.DataFrame({'local_path': fns})
    df['tile_num'] = df.local_path.str.extract(rf'SAR_S1_{year}_(\d+)\.tif$')[0].astype(int)
    df['mtime'] = [os.path.getmtime(f) for f in df.local_path]
    df['s3_path'] = df.local_path.str.replace('/projects/my-private-bucket', f's3://maap-ops-workspace/{USER}', regex=False)
    df['manifest'] = df.local_path.str.replace('.tif', '_manifest.json', regex=False)
    return df.sort_values('mtime').drop_duplicates('tile_num', keep='last').reset_index(drop=True)

def rebuild_tindex(identifier, year):
    """Back up, rebuild the tindex (CSV + GPKG with tile polygons) from ALL outputs for the identifier, verify the count.
    Columns follow the other tindexes (tile_num, s3_path, local_path) so mosaiclib/SAR_TINDEX_FN_DICT can point to it."""
    tp = tindex_paths(identifier)
    found = find_outputs(identifier, year)
    if found.empty:
        print(f'  no outputs under {DPS_OUTPUT_ROOT}/{identifier}; skipping')
        return None
    n_before = len(read_tindex_tiles(identifier))
    stamp = pd.Timestamp.now().strftime('%Y%m%d%H%M%S')
    bdir = f"{tp['dir']}/backup_{stamp}"
    for fn in (tp['csv'], tp['gpkg']):
        if os.path.exists(fn):
            os.makedirs(bdir, exist_ok=True)
            shutil.copy2(fn, bdir)
    os.makedirs(tp['dir'], exist_ok=True)
    found = found.assign(year=year, identifier=identifier, version=VERSION)
    found.drop(columns='mtime').to_csv(tp['csv'], index=False)
    gdf = boreal_tiles[['tile_num', 'geometry']].merge(found.drop(columns='mtime'), on='tile_num')
    gdf.to_file(tp['gpkg'], driver='GPKG')
    n_after = len(found)
    flag = 'OK' if n_after >= n_before else '!! SHRANK — restore with restore_tindex(identifier, backup_dir)'
    print(f'  tiles before {n_before} -> after {n_after}  {flag}   (backup: {bdir if n_before else "none"})')
    return {'identifier': identifier, 'n_before': n_before, 'n_after': n_after, 'tindex_csv': tp['csv'], 'backup_dir': bdir}

def restore_tindex(identifier, backup_dir):
    for fn in glob.glob(f'{backup_dir}/*'):
        shutil.copy2(fn, tindex_paths(identifier)['dir'])
    print('restored', identifier, 'from', backup_dir)

## 2. Status: which tiles each year has and is missing

In [ ]:
TILES = sorted(set(int(t) for t in (TILE_SET if TILE_SET is not None else boreal_tiles.tile_num)))
HAVE = {y: read_tindex_tiles(i) for y, i in IDENTIFIERS.items()}
status = pd.DataFrame([{'year': y, 'identifier': IDENTIFIERS[y], 'n_target': len(TILES),
                        'n_have': len(set(TILES) & HAVE[y]), 'n_missing': len(set(TILES) - HAVE[y])} for y in YEAR_LIST])
status

In [ ]:
# Optional: map of missing tiles for one year
SHOW_MAP_YEAR = YEAR_LIST[0]
boreal_tiles[boreal_tiles.tile_num.isin(set(TILES) - HAVE[SHOW_MAP_YEAR])].explore(color='red', tooltip='tile_num')

## 3. Fails: latest job per tile from the saved submission logs
Only tiles whose **most recent** job for the identifier failed (or never got a job id), and that are still not in the tindex.
Cached to JSON so `DO_FAILS` works when you come back without re-querying (`REFRESH_FAILS = False`).

In [ ]:
FAILS_JSON = f"{PLAN_DIR}/fails_{ALGO_ID}_{VERSION}_{SUB_IDENTIFIER}.json"
if DO_FAILS and not REFRESH_FAILS and os.path.exists(FAILS_JSON):
    FAILED = {k: set(v) for k, v in json.load(open(FAILS_JSON)).items()}
    print('loaded cached fails from', FAILS_JSON)
elif DO_FAILS:
    FAILED, job_tables = {}, {}
    for y, ident in IDENTIFIERS.items():
        jobs = latest_jobs(ident)
        if jobs.empty:
            FAILED[y] = set(); print(f'{ident}: no submission logs'); continue
        jt = job_status_table(jobs)
        job_tables[y] = jt
        bad = jt.status.isin(['Failed', 'SubmitFailed', 'Offline', 'Deleted'])
        FAILED[y] = set(jt.tile_num[bad].astype(int)) - HAVE[y]
        print(f'{ident}: latest jobs {len(jt)} | ' + ', '.join(f'{k} {v}' for k, v in jt.status.value_counts().items())
              + f' | failed & not in tindex: {len(FAILED[y])}')
    json.dump({y: sorted(v) for y, v in FAILED.items()}, open(FAILS_JSON, 'w'))
else:
    FAILED = {y: set() for y in YEAR_LIST}
    print('DO_FAILS = False: fails not computed')

## 4. Coverage: OPERA RTC-S1 granules per tile and year
Coverage of OPERA RTC-S1 (`OPERA_L2_RTC-S1_V1`) varies by region and year (e.g. Sentinel-1B ended in Dec 2021, so 2022–2024 have
fewer passes). Tiles with zero granules in a year are marked not runnable, so they don't fail repeatedly; the granule counts also
show where seasonal means rest on few passes. The first run queries CMR for every tile (a few minutes); later
runs read the cache.

In [ ]:
COVERAGE = {}
if CHECK_OPERA_COVERAGE:
    for y in YEAR_LIST:
        COVERAGE[y] = opera_coverage(y, TILES)
        h = pd.Series(COVERAGE[y])
        print(f'{y}: {int((h > 0).sum())} tiles with OPERA RTC-S1 granules, {int((h == 0).sum())} without '
              f'(median {int(h[h > 0].median()) if (h > 0).any() else 0} granules per covered tile)')
else:
    print('CHECK_OPERA_COVERAGE = False: coverage not checked')

## 5. Plan: what will be submitted
Tiles that cannot run (bbox crossing the antimeridian in lon/lat, no OPERA granules that year) are listed as not runnable and are **not** submitted.
Review this table before setting `DRY_RUN = False`.

In [ ]:
if not (DO_MISSING or DO_FAILS) and not CONFIRM_FULL_RUN:
    raise ValueError('Both DO_MISSING and DO_FAILS are False (full re-run of TILE_SET). Set CONFIRM_FULL_RUN = True if intended.')

plan = pd.concat([build_plan(y, IDENTIFIERS[y], TILES, HAVE[y], FAILED.get(y, set()), COVERAGE.get(y)) for y in YEAR_LIST], ignore_index=True)
plan = plan.astype({'tile_num': int, 'is_missing': bool, 'is_failed': bool, 'in_tindex': bool, 'runnable': bool})
plan_summary = (plan.groupby(['year', 'identifier'])
                    .agg(n_selected=('tile_num', 'size'), n_missing=('is_missing', 'sum'), n_failed=('is_failed', 'sum'),
                         n_not_runnable=('runnable', lambda s: int((~s).sum())), n_to_submit=('runnable', 'sum'))
                    .reset_index())
PLAN_FN = f"{PLAN_DIR}/plan_{ALGO_ID}_{SUB_IDENTIFIER}_{MODE}_{pd.Timestamp.now():%Y%m%d%H%M}.csv"
plan.to_csv(PLAN_FN, index=False)
print('plan saved:', PLAN_FN, '| total to submit:', int(plan.runnable.sum()))
plan_summary

In [ ]:
# Not-runnable tiles and why
plan[~plan.runnable].groupby(['year', 'reason_not_runnable']).tile_num.apply(lambda s: sorted(s)).reset_index()

In [ ]:
# Parameters of the first tile to submit, exactly as they will be sent
first = plan[plan.runnable].head(1)
if len(first):
    t, y = int(first.tile_num.iat[0]), first.year.iat[0]
    print(json.dumps(get_in_param_dict(t, y), indent=1), '\ngrid (cols x rows):', tile_bbox(t)[1])

## 6. Submit (only if `DRY_RUN = False`)

In [ ]:
SUBMITTED = {}
n_total = int(plan.runnable.sum())
if DRY_RUN:
    print(f'DRY_RUN: would submit {n_total} jobs. Set DRY_RUN = False in section 1 and re-run from the top.')
elif n_total > MAX_JOBS:
    raise ValueError(f'{n_total} jobs > MAX_JOBS={MAX_JOBS}. Raise MAX_JOBS if intended.')
else:
    for y, ident in IDENTIFIERS.items():
        tiles = plan[(plan.year == y) & plan.runnable].tile_num.tolist()
        print(f'\n{ident}: {len(tiles)} jobs')
        if tiles:
            SUBMITTED[y] = submit_identifier(ident, y, tiles)

## 7. Assess the submission just made
Re-run this cell until nothing is `Accepted`/`Running`. To assess an older run in a new session, set `DO_FAILS = True` and run
section 3, which reads the saved logs.

In [ ]:
if SUBMITTED:
    assess = pd.concat([job_status_table(df) for df in SUBMITTED.values()], ignore_index=True)
    display(assess.groupby(['IDENTIFIER', 'status']).size().unstack(fill_value=0))
else:
    print('Nothing submitted in this session.')

## 8. Build the tindex (after jobs finish)
Scans **all** DPS output folders for each identifier (`dps_output/<alg>/<version>/<identifier>/YYYY/MM/DD/...`), keeps the newest
file per tile, backs up the existing tindex first, and reports before/after tile counts. Then go back to section 2.

In [ ]:
DO_REBUILD_TINDEX = True
if DO_REBUILD_TINDEX:
    rebuilt = [rebuild_tindex(i, y) for y, i in IDENTIFIERS.items()]
    display(pd.DataFrame([r for r in rebuilt if r]))

---
## Appendix: reference calls (not part of the run)

In [ ]:
SHOW_REFERENCE = True
if SHOW_REFERENCE:
    print([a for a in maap.listAlgorithms().json()['algorithms'] if ALGO_ID in a['type']])
    print(maap.getQueues().json()['queues'])
    # Register the algorithm (only after changing the yml / code, once committed to remote main):
    # maap.register_algorithm_from_yaml_file(f'{S1_REPO_PATH}/algorithm_config.yml').text
    # One test job (tile 3364, the README example) before a large run:
    # r = maap.submitJob(identifier=f'S1_2020/{SUB_IDENTIFIER}_test', algo_id=ALGO_ID, version=VERSION, queue=WORKER_TYPE, **get_in_param_dict(3364, '2020')); print(r.id, r.status)
    print(json.dumps(get_in_param_dict(TILES[0], YEAR_LIST[0]), indent=1, default=str))